In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# ==============================================================================
# 1. PARÁMETROS FÍSICOS Y TEÓRICOS
# ==============================================================================
T_A_C = 15.0  # Temperatura ambiente (°C)
sigma_T = 0.1  # Precisión de los sensores (°C)

# Propiedades del material
r_int, r_ext = 0.00582, 0.0125  # Radios interno y externo (m)
D_ext = 2 * r_ext  # Diámetro exterior (m)
k_A = 0.04  # Conductividad térmica del aislante (W/m K)
epsilon = 0.90  # Emisividad de la espuma de poliuretano
sigma_SB = 5.67e-8  # Constante de Stefan-Boltzmann (W/m^2 K^4)

R_cond = np.log(r_ext / r_int) / (2 * np.pi * k_A)

# ==============================================================================
# 2. CARGA DE DATOS Y LIMPIEZA DEL CSV 
# ==============================================================================
nombre_archivo = "Prueba 4 - Prueba 4.csv"  # Ruta del csv

datos_limpios = []
try:
    with open(nombre_archivo, "r", encoding="utf-8", errors="ignore") as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith("#"):
                continue
            partes = [p.strip() for p in linea.replace(";", ",").split(",")]
            if len(partes) >= 3:
                try:
                    t = float(partes[0])
                    pos = float(partes[1])
                    temp_C = float(partes[2])
                    datos_limpios.append([t, pos, temp_C])
                except ValueError:
                    continue
except FileNotFoundError:
    raise FileNotFoundError(f"No se encontró el archivo '{nombre_archivo}'.")

df = pd.DataFrame(datos_limpios, columns=["tiempo", "posicion", "temperatura"])

if df.empty:
    raise ValueError("El DataFrame quedó vacío después de limpiar el CSV.")

if df["posicion"].max() > 1.0:
    df["posicion"] = df["posicion"] / 100.0

df = df.drop_duplicates(subset=["tiempo", "posicion"], keep="last")
df_pivot = df.pivot_table(
    index="tiempo", columns="posicion", values="temperatura"
).reset_index()

cols_sensores = [c for c in df_pivot.columns if c != "tiempo"]

# ==============================================================================
# 3. IDENTIFICACIÓN DE t_o
# ==============================================================================
# Se identifica el sensor más cercano a la fuente
col_f = max(cols_sensores, key=lambda c: df_pivot[c].max())

#Se busca automáticamente t_o,donde el sensor alcanza el 98% de su temperatura máxima
T_max_sensor = df_pivot[col_f].max()
idx_meseta = df_pivot[df_pivot[col_f] >= (T_max_sensor - 0.5)]

# Se define t_o como la mediana del intervalo de tiempo
t_0 = idx_meseta["tiempo"].median()

# ==============================================================================
# 4. CÁLCULO DE h_conv, h_rad,T_s,RESISTENCIAS Y SALTOS TÉRMICOS
# ==============================================================================
idx_estacionario = df_pivot["tiempo"] >= idx_meseta["tiempo"].min()
T_s_C = df_pivot.loc[idx_estacionario, cols_sensores].mean().mean()
delta_Ts = T_s_C - T_A_C

h_conv = 1.42 * (delta_Ts / D_ext) ** 0.25
T_A_K = T_A_C + 273.15
h_rad = 4 * epsilon * sigma_SB * (T_A_K**3)

R_paralelo = 1.0 / ((h_conv + h_rad) * 2 * np.pi * r_ext)
R_total = R_cond + R_paralelo

# Salto térmico máximo
T_max_global_C = df_pivot[cols_sensores].max().max()
delta_T_exp = T_max_global_C - T_A_C
sigma_delta_T_exp = sigma_T

# Salto térmico en el tiempo t_0
idx_t0 = (df_pivot["tiempo"] - t_0).abs().idxmin()
T_f_C = df_pivot.loc[idx_t0, col_f]

delta_T_teorico = T_f_C - T_A_C
sigma_delta_T_teor = np.sqrt(sigma_T**2 + sigma_T**2)

desvio_porcentual = (
    abs(delta_T_exp - delta_T_teorico) / delta_T_teorico
) * 100.0

# Conversión a Kelvin
T_A_Kelvin = T_A_C + 273.15
T_max_global_Kelvin = T_max_global_C + 273.15
T_f_Kelvin = T_f_C + 273.15

# ==============================================================================
# 5. RESULTADOS
# ==============================================================================
print("======================================================================")
print("     COMPARACIÓN DE PARÁMETROS Y TEMPERATURAS            ")
print("======================================================================")
print(f"Temperatura Ambiente (T_A):           {T_A_Kelvin:.2f} K")
print(
    f"Temperatura Máxima Global (T_max):    {T_max_global_Kelvin:.2f} ±"
    f" {sigma_T:.2f} K"
)
print(
    f"Temperatura en t_0 (T_f):             {T_f_Kelvin:.2f} ± {sigma_T:.2f} K"
)
print("----------------------------------------------------------------------")
print(
    f"Salto Térmico Experimental (ΔT_exp):  {delta_T_exp:.2f} ±"
    f" {sigma_delta_T_exp:.2f} K"
)
print(
    f"Salto Térmico Teórico (ΔT_teor):      {delta_T_teorico:.2f} ±"
    f" {sigma_delta_T_teor:.2f} K"
)
print(f"Desviación porcentual:                {desvio_porcentual:.2f} %")
print("======================================================================")